<a href="https://colab.research.google.com/github/Tsaraban/Tsaraban_Capstone_Project/blob/main/data_convert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **This file has been used to convert drugbank xml dataset to csv file**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os


folder = "/content/drive/MyDrive/4th year Coursework/Capstone Project/"

for f in os.listdir(folder):
    print(f)

data_convert.ipynb
full_database.xml


In [ ]:
!pip install pandas lxml tqdm

import pandas as pd
from lxml import etree
from tqdm import tqdm

NS = "http://www.drugbank.ca"
NS_MAP = {"db": NS}

def tag(name):
    return f"{{{NS}}}{name}"

def parse_interactions(xml_path):
    records = []
    context = etree.iterparse(xml_path, events=("end",), tag=tag("drug"))

    for event, drug_elem in tqdm(context):
        if drug_elem.getparent() is None or drug_elem.getparent().tag == tag("drugbank"):
            drug_id_elem = drug_elem.find("db:drugbank-id[@primary='true']", NS_MAP)
            drug_id = drug_id_elem.text if drug_id_elem is not None else None

            drug_name_elem = drug_elem.find("db:name", NS_MAP)
            drug_name = drug_name_elem.text if drug_name_elem is not None else None

            interactions_elem = drug_elem.find("db:drug-interactions", NS_MAP)
            if interactions_elem is not None:
                for interaction in interactions_elem.findall("db:drug-interaction", NS_MAP):
                    inter_id   = getattr(interaction.find("db:drugbank-id", NS_MAP), "text", None)
                    inter_name = getattr(interaction.find("db:name", NS_MAP), "text", None)
                    desc       = getattr(interaction.find("db:description", NS_MAP), "text", None)

                    records.append({
                        "drug_id": drug_id,
                        "drug_name": drug_name,
                        "interacting_drug_id": inter_id,
                        "interacting_drug_name": inter_name,
                        "description": desc,
                    })

        drug_elem.clear()

    df = pd.DataFrame(records)
    print(f"✓ {len(df):,} interactions from {df['drug_id'].nunique():,} drugs")
    return df

xml_path = "/content/drive/MyDrive/4th year Coursework/Capstone Project/full_database.xml"
df = parse_interactions(xml_path)
df.head()

1014340it [02:27, 6864.75it/s]


✓ 2,911,156 interactions from 4,631 drugs


,drug_id,drug_name,interacting_drug_id,interacting_drug_name,description
0,DB00001,Lepirudin,DB06605,Apixaban,Apixaban may increase the anticoagulant activi...
1,DB00001,Lepirudin,DB06695,Dabigatran etexilate,Dabigatran etexilate may increase the anticoag...
2,DB00001,Lepirudin,DB01254,Dasatinib,The risk or severity of bleeding and hemorrhag...
3,DB00001,Lepirudin,DB01609,Deferasirox,The risk or severity of gastrointestinal bleed...
4,DB00001,Lepirudin,DB01586,Ursodeoxycholic acid,The risk or severity of bleeding and bruising ...


Save to drive

In [ ]:
output_path = "/content/drive/MyDrive/4th year Coursework/Capstone Project/interactions.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to /content/drive/MyDrive/4th year Coursework/Capstone Project/interactions.csv
